<a href="https://colab.research.google.com/github/naveeak/nano-gpt-impl/blob/my-impl/nano_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-06-20 10:46:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-06-20 10:46:27 (46.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [19]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
dropout = 0.2
n_head= 6
n_layer = 6

# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        embed = self.token_embedding_table(idx) # (B,T,C)
        pos_embed = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = embed + pos_embed # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx[:, -block_size:])
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

class MultiHeadAttention(nn.Module):
  """ Mulit head attention"""

  def __init__(self, nums_head, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(nums_head)])
    self.proj = nn.Linear(n_embd, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))
    return out

class Head(nn.Module):
  "on head of self-attentin"

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,-1) * C**-0.5
    wei = wei.masked_fill(self.tril[:T,:T] == 0 , float('-inf'))
    wei = F.softmax(wei, dim = -1)
    wei = self.dropout(wei)

    v = self.value(x)
    out = wei @ v

    return out

class FeedForward(nn.Module):
  "simple feed forward layer followed by non linearity"

  def  __init__(self, n_embd) :
     super().__init__()
     self.net = nn.Sequential(
         nn.Linear(n_embd, 4*n_embd), # 4 times of mebd layer
         nn.ReLU(),
         nn.Linear(4* n_embd, n_embd),
         nn.Dropout(dropout)
     )
  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  "transformor block"

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd//n_head;
    self.sa_head = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa_head(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))

    return x


model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

step 0: train loss 4.2849, val loss 4.2823
step 500: train loss 2.0136, val loss 2.0992
step 1000: train loss 1.6019, val loss 1.7824
step 1500: train loss 1.4411, val loss 1.6396
step 2000: train loss 1.3430, val loss 1.5728
step 2500: train loss 1.2814, val loss 1.5372
step 3000: train loss 1.2275, val loss 1.5126
step 3500: train loss 1.1829, val loss 1.4920
step 4000: train loss 1.1477, val loss 1.4902
step 4500: train loss 1.1102, val loss 1.4836

But with prison: I will seek think you,
And some me are seconce for ano right
Our lord Tarpeians; good night
I'll revenge him down to my brother's blood youth.

SICINIUS:
What companions them?

BRUTUS:
Let's hear him chrict: come to desire up;
And soon stays he find his face, steep'd with statue.
Now to it there sirs o' the plights, our fieldessine,
May not suffer the crowned purgar i' peace.
They farewell well have hem hence with this come to health:
Yet black you! We have fright much more suns


In [20]:
print(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))


I am should dedlors to meet you go
Receive me in richly.

Clown:
Neither, fellow, truly.

Second Servingman:
A grass. Wearing claim your consuls, your fathers,
You dread me with lose is land?

ESCALUS:
The green, edign.

Second Plantagenet:
And sweet him greeting to as faces for yourself.
This is now by my life, you take your whoresome
Cp wafrant, were my name: I have you weiped,
That wich you should smook not you have vault,
The know of your lovery parents, be strike to meet
The correction of English blush.

KING HENRY VI:
Cour back, nor better brother,--as it were the king
Likely order from that sea against love refument,
Which some trives none to thee mark of you;
Sick against and pay you but bring, we are.

RICHARD:
No, by consisting them ghave very chance:
Unhoperless carry may by us,
By Bumy owe is clear-gage?

JULIET:
No margant me concers!
Farewell, but smiled me,
Dors Vlutchange successits and nature,
I hold the wim acred for my spacious solvice.

JULIET:
Escalue, I pray too 

In [ ]:
# need to save the model parametere in some place so that it will not issed and resume from same point

